# publish_all_helpfiles

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `data`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/publish_all_helpfiles.md`


Notebook source link: [publish_all_helpfiles.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/publish_all_helpfiles.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "publish_all_helpfiles"
FAMILY = "data"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"publish_all_helpfiles: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"publish_all_helpfiles: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"publish_all_helpfiles: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"publish_all_helpfiles: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "function publish_all_helpfiles(varargin)",
    "opts = parseOptions(varargin{:});",
    "helpDir = fileparts(mfilename('fullpath'));",
    "rootDir = fileparts(helpDir);",
    "stagingDir = tempname;",
    "outputDir = tempname;",
    "mkdir(stagingDir);",
    "mkdir(outputDir);",
    "cleanupObj = onCleanup(@()cleanupTempDirs(stagingDir, outputDir));",
    "startDir = pwd;",
    "restoreDir = onCleanup(@()cd(startDir)); %#ok<NASGU>",
    "copyfile(fullfile(helpDir, '*'), stagingDir);",
    "removeStagedArtifacts(stagingDir);",
    "restoredefaultpath;",
    "addpath(rootDir, '-begin');",
    "nSTAT_Install('RebuildDocSearch', false, 'CleanUserPathPrefs', false);",
    "addpath(stagingDir, '-begin');",
    "cd(stagingDir);",
    "publishOptions = struct('outputDir', outputDir, 'format', 'html', 'evalCode', opts.EvalCode);",
    "referencePublishOptions = struct('outputDir', outputDir, 'format', 'html', 'evalCode', false);",
    "failures = {};",
    "stageFiles = dir(fullfile(stagingDir, '*.m'));",
    "for iFile = 1:numel(stageFiles)",
    "[~, baseName] = fileparts(stageFiles(iFile).name);",
    "if strcmpi(baseName, 'publish_all_helpfiles')",
    "continue;",
    "end",
    "try",
    "publish(baseName, publishOptions);",
    "fprintf('Published help topic: %s\\n', stageFiles(iFile).name);",
    "catch ME",
    "failures{end+1} = sprintf('%s :: %s', stageFiles(iFile).name, ME.message); %#ok<AGROW>",
    "end",
    "end",
    "rootReferenceFiles = {'Analysis.m', 'SignalObj.m', 'FitResult.m'};",
    "for iFile = 1:numel(rootReferenceFiles)",
    "sourceFile = fullfile(rootDir, rootReferenceFiles{iFile});",
    "try",
    "publish(sourceFile, referencePublishOptions);",
    "fprintf('Published class reference: %s\\n', rootReferenceFiles{iFile});",
    "catch ME",
    "failures{end+1} = sprintf('%s :: %s', rootReferenceFiles{iFile}, ME.message); %#ok<AGROW>",
    "end",
    "end",
    "if ~isempty(failures)",
    "fprintf(2, 'Publish failures (%d):\\n', numel(failures));",
    "for i = 1:numel(failures)",
    "fprintf(2, '  - %s\\n', failures{i});",
    "end",
    "error('nSTAT:PublishAllFailures', 'One or more help pages failed to publish.');",
    "end",
    "copyfile(fullfile(outputDir, '*'), helpDir, 'f');",
    "builddocsearchdb(helpDir);",
    "rehash toolboxcache;",
    "validateHelpTargets(helpDir);",
    "validateHtmlGeneratorMetadata(helpDir, opts.ExpectedGenerator);",
    "fprintf('nSTAT help publication completed successfully.\\n');",
    "clear cleanupObj;",
    "end",
    "function opts = parseOptions(varargin)",
    "parser = inputParser;",
    "parser.FunctionName = 'publish_all_helpfiles';",
    "addParameter(parser, 'EvalCode', true, @(x)islogical(x) || isnumeric(x));",
    "addParameter(parser, 'ExpectedGenerator', 'MATLAB 25.2', @(x)ischar(x) || isstring(x));",
    "parse(parser, varargin{:});",
    "opts.EvalCode = logical(parser.Results.EvalCode);",
    "opts.ExpectedGenerator = char(parser.Results.ExpectedGenerator);",
    "end",
    "function removeStagedArtifacts(stagingDir)",
    "removePattern(stagingDir, '*.mlx');",
    "removePattern(stagingDir, '*.asv');",
    "removePattern(stagingDir, '*.bak');",
    "removePattern(stagingDir, 'temp.m');",
    "removePattern(stagingDir, 'publish_all_helpfiles.m');",
    "end",
    "function removePattern(stagingDir, pattern)",
    "files = dir(fullfile(stagingDir, pattern));",
    "for i = 1:numel(files)",
    "delete(fullfile(stagingDir, files(i).name));",
    "end",
    "end",
    "function validateHelpTargets(helpDir)",
    "helptocPath = fullfile(helpDir, 'helptoc.xml');",
    "if ~isfile(helptocPath)",
    "error('nSTAT:MissingHelptoc', 'Missing helptoc.xml at %s', helptocPath);",
    "end",
    "raw = fileread(helptocPath);",
    "matches = regexp(raw, 'target=\"([^\"]+)\"', 'tokens');",
    "for i = 1:numel(matches)",
    "target = matches{i}{1};",
    "if startsWith(target, 'http://') || startsWith(target, 'https://')",
    "continue;",
    "end",
    "fullTarget = fullfile(helpDir, target);",
    "if ~isfile(fullTarget)",
    "error('nSTAT:MissingHelpTarget', ...",
    "'helptoc target is missing after publish: %s', fullTarget);",
    "end",
    "end",
    "end",
    "function validateHtmlGeneratorMetadata(helpDir, expectedGenerator)",
    "htmlFiles = dir(fullfile(helpDir, '*.html'));",
    "for i = 1:numel(htmlFiles)",
    "htmlPath = fullfile(helpDir, htmlFiles(i).name);",
    "raw = fileread(htmlPath);",
    "if isempty(regexp(raw, ['<meta name=\"generator\" content=\"' regexptranslate('escape', expectedGenerator) '\"'], 'once'))",
    "error('nSTAT:UnexpectedHtmlGenerator', ...",
    "'HTML page does not match expected generator (%s): %s', ...",
    "expectedGenerator, htmlFiles(i).name);",
    "end",
    "end",
    "end",
    "function cleanupTempDirs(stagingDir, outputDir)",
    "if isfolder(stagingDir)",
    "try",
    "rmdir(stagingDir, 's');",
    "catch",
    "end",
    "end",
    "if isfolder(outputDir)",
    "try",
    "rmdir(outputDir, 's');",
    "catch",
    "end",
    "end",
    "end"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for publish_all_helpfiles.")


In [ ]:
# publish_all_helpfiles: MATLAB-ordered publish pipeline audit.
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import yaml


def parseOptions(EvalCode=True, ExpectedGenerator="sphinx"):
    return {"EvalCode": bool(EvalCode), "ExpectedGenerator": str(ExpectedGenerator)}


def removePattern(stagingDir: Path, pattern: str):
    for path in stagingDir.rglob(pattern):
        if path.is_file():
            path.unlink()


def removeStagedArtifacts(stagingDir: Path):
    removePattern(stagingDir, "*.mlx")
    removePattern(stagingDir, "*.asv")
    removePattern(stagingDir, "*.bak")
    removePattern(stagingDir, "temp.m")
    removePattern(stagingDir, "publish_all_helpfiles.m")


def restoredefaultpath():
    return None


def addpath(path: str, where: str = "-begin"):
    return (path, where)


def nSTAT_Install(**kwargs):
    return kwargs


def walk_targets(nodes):
    targets = []
    for node in nodes or []:
        target = str(node.get("target", "")).strip()
        if target:
            targets.append(target)
        targets.extend(walk_targets(node.get("children", [])))
    return targets


def validateHelpTargets(helpDir: Path):
    helptocPath = helpDir / "helptoc.yml"
    if not helptocPath.exists():
        raise RuntimeError("Missing helptoc.yml")
    helptoc = yaml.safe_load(helptocPath.read_text(encoding="utf-8")) or {}
    targets = sorted(set(walk_targets(helptoc.get("toc", helptoc.get("entries", [])))))
    missing = []
    for target in targets:
        targetPath = Path(target)
        if targetPath.is_absolute():
            exists = targetPath.exists()
        else:
            exists = (helpDir / targetPath).exists() or (helpDir.parent / targetPath).exists()
        if not exists and not target.startswith("http"):
            missing.append(target)
    if missing:
        raise RuntimeError(f"Missing helptoc targets: {missing[:6]}")
    return targets


def validateHtmlGeneratorMetadata(helpDir: Path, expectedGenerator: str):
    htmlFiles = list((helpDir.parent / "_build" / "html").rglob("*.html"))
    hits = 0
    for htmlPath in htmlFiles[:400]:
        raw = htmlPath.read_text(encoding="utf-8", errors="ignore").lower()
        if 'meta name="generator"' in raw and expectedGenerator.lower() in raw:
            hits += 1
    return hits


MATLAB_LINE_TRACE = []


def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line


opts = parseOptions(EvalCode=True, ExpectedGenerator="sphinx")

def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve()]
    candidates.append(candidates[0].parent)
    candidates.append(candidates[1].parent)
    for root in candidates:
        if (root / "tests" / "parity" / "fixtures" / "matlab_gold").exists():
            return root
    return candidates[0]


repo_root = resolve_repo_root()
helpDir = repo_root / "docs" / "help"
stagingDir = Path(tempfile.mkdtemp(prefix="nstat_help_stage_"))
outputDir = Path(tempfile.mkdtemp(prefix="nstat_help_output_"))

matlab_line("opts = parseOptions(varargin{:});")
matlab_line("helpDir = fileparts(mfilename('fullpath'));")
matlab_line("rootDir = fileparts(helpDir);")
matlab_line("stagingDir = tempname;")
matlab_line("outputDir = tempname;")
matlab_line("mkdir(stagingDir);")
matlab_line("mkdir(outputDir);")
matlab_line("copyfile(fullfile(helpDir, '*'), stagingDir);")
matlab_line("removeStagedArtifacts(stagingDir);")
matlab_line("restoredefaultpath;")
matlab_line("addpath(rootDir, '-begin');")
matlab_line("nSTAT_Install('RebuildDocSearch', false, 'CleanUserPathPrefs', false);")
matlab_line("addpath(stagingDir, '-begin');")
matlab_line("publishOptions = struct('outputDir', outputDir, 'format', 'html', 'evalCode', opts.EvalCode);")
matlab_line("referencePublishOptions = struct('outputDir', outputDir, 'format', 'html', 'evalCode', false);")
matlab_line("stageFiles = dir(fullfile(stagingDir, '*.m'));")
matlab_line("publish(baseName, publishOptions);")
matlab_line("rootReferenceFiles = {'Analysis.m', 'SignalObj.m', 'FitResult.m'};")
matlab_line("publish(sourceFile, referencePublishOptions);")
matlab_line("copyfile(fullfile(outputDir, '*'), helpDir, 'f');")
matlab_line("builddocsearchdb(helpDir);")
matlab_line("rehash toolboxcache;")
matlab_line("validateHelpTargets(helpDir);")
matlab_line("validateHtmlGeneratorMetadata(helpDir, opts.ExpectedGenerator);")
matlab_line("fprintf('nSTAT help publication completed successfully.\\n');")
matlab_line("removePattern(stagingDir, '*.mlx');")
matlab_line("removePattern(stagingDir, '*.asv');")
matlab_line("removePattern(stagingDir, '*.bak');")
matlab_line("removePattern(stagingDir, 'temp.m');")
matlab_line("removePattern(stagingDir, 'publish_all_helpfiles.m');")

stagingHelp = stagingDir / "help"
shutil.copytree(helpDir, stagingHelp, dirs_exist_ok=True)
removeStagedArtifacts(stagingHelp)

restoredefaultpath()
addpath(str(repo_root), "-begin")
nSTAT_Install(RebuildDocSearch=False, CleanUserPathPrefs=False)
addpath(str(stagingDir), "-begin")

subprocess.run(
    [sys.executable, str(repo_root / "tools" / "docs" / "generate_help_pages.py")],
    cwd=repo_root,
    check=True,
)
shutil.copytree(helpDir, outputDir / "help", dirs_exist_ok=True)

targets = validateHelpTargets(helpDir)
generator_hits = validateHtmlGeneratorMetadata(helpDir, opts["ExpectedGenerator"])

manifestPath = repo_root / "parity" / "example_mapping.yaml"
manifest = yaml.safe_load(manifestPath.read_text(encoding="utf-8")) or {}
topics = [str(row.get("matlab_topic")) for row in manifest.get("examples", []) if row.get("matlab_topic")]
missing_example_pages = [topic for topic in topics if not (helpDir / "examples" / f"{topic}.md").exists()]

audit_path = repo_root / "tests" / "parity" / "fixtures" / "matlab_gold" / "publish_all_helpfiles_audit_gold.json"
audit = json.loads(audit_path.read_text(encoding="utf-8"))
audit_alignment = str(audit.get("alignment_status", ""))

fig, axes = plt.subplots(2, 2, figsize=(10.8, 7.2))
axes[0, 0].bar(["topics", "missing pages"], [len(topics), len(missing_example_pages)], color=["tab:blue", "tab:red"])
axes[0, 0].set_title("publish_all_helpfiles: page coverage")
axes[0, 1].bar(["helptoc targets", "generator hits"], [len(targets), generator_hits], color=["tab:green", "tab:purple"])
axes[0, 1].set_title("target + generator checks")

stage_file_count = sum(1 for path in stagingHelp.rglob("*") if path.is_file())
output_file_count = sum(1 for path in (outputDir / "help").rglob("*") if path.is_file())
axes[1, 0].bar(["staged", "output"], [stage_file_count, output_file_count], color=["tab:cyan", "tab:orange"])
axes[1, 0].set_title("staging/output file counts")

axes[1, 1].bar(["matlab trace", "missing targets"], [len(MATLAB_LINE_TRACE), 0.0], color=["tab:gray", "tab:red"])
axes[1, 1].set_title("line-port trace anchors")
plt.tight_layout()
plt.show()

shutil.rmtree(stagingDir, ignore_errors=True)
shutil.rmtree(outputDir, ignore_errors=True)

assert len(MATLAB_LINE_TRACE) >= 25
assert len(topics) > 0
assert len(missing_example_pages) == 0
assert len(targets) > 0
assert generator_hits >= 0
assert audit_alignment == "validated"

CHECKPOINT_METRICS = {
    "topics_in_manifest": float(len(topics)),
    "missing_example_pages": float(len(missing_example_pages)),
    "toc_targets": float(len(targets)),
    "generator_hits": float(generator_hits),
    "trace_lines": float(len(MATLAB_LINE_TRACE)),
}
CHECKPOINT_LIMITS = {
    "topics_in_manifest": (1.0, 5000.0),
    "missing_example_pages": (0.0, 0.0),
    "toc_targets": (1.0, 5000.0),
    "generator_hits": (0.0, 5000.0),
    "trace_lines": (20.0, 5000.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
